# Bayesian dual-clock spatio-temporal point processes
### Simulation, Bayesian fitting and Cascadia tectonic tremor analysis

Computational companion to the draft manuscript. **No real-data estimates are embedded**. Start with SMOKE; then PILOT and RESEARCH, with adequate diagnostics. The full research run is computationally substantial, so use separate replicate batches and persistent storage. All figures, tables, posterior draws and provenance are written to a result directory and downloaded as one ZIP.

## 1. Install and configure
CPU runtime is sufficient. The notebook embeds both source modules and does not require a separate code upload.

In [ ]:
# Install only missing dependencies. Restart runtime if Colab requests it.
import importlib.util, subprocess, sys
required = {'numpy':'numpy','scipy':'scipy','pandas':'pandas','numba':'numba','matplotlib':'matplotlib','arviz':'arviz','pyproj':'pyproj'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from pathlib import Path
import json, os


## 2. Statistical implementation (editable source)
The primary likelihood is the manuscript's normalized truncated Gaussian kernel and analytic compensator, with an explicit-duration negative-binomial path prior. The sampler uses valid direct path Metropolis updates, not PMMH. Review the source or collapse the two large cells.

In [ ]:
%%writefile dual_clock.py
"""Dual-clock Hawkes: reproducible conditional likelihood, simulation and MCMC.
Units: days, kilometres. State codes 0=pause, 1=forward, 2=reverse.
The target is the *finite-memory* model when memory < infinity. No event subsampling.
"""
from dataclasses import dataclass
from pathlib import Path
import math, time, json, hashlib, warnings
import numpy as np
import pandas as pd
from scipy.special import expit, logit, logsumexp, ndtr
from scipy.stats import nbinom, truncnorm
from numba import njit
import matplotlib.pyplot as plt
import arviz as az

RHO=.98
MODELS=('M0','M1','M2','M3','M4','M5','M5slow')
LABELS={'M0':'State Poisson','M1':'Fixed Hawkes','M2':'Active Hawkes','M3':'Rate-only Hawkes','M4':'Two-state clock','M5':'Dual clock','M5slow':'Slow-state clock'}
# For rate-only models the state labels do not represent transport.

def states(model):
    return 1 if model in ('M1','M2') else 2 if model=='M4' else 3

def factors(model):
    if model in ('M0','M1'): return np.zeros(3),np.zeros(3)
    if model in ('M2','M3','S3'): return np.ones(3),np.ones(3)
    if model in ('M5slow','S7'): return np.array([.1,1.,1.]),np.array([0.,1.,-1.])
    return np.array([0.,1.,1.]),np.array([0.,1.,-1.])

@dataclass
class Data:
    t: np.ndarray
    xy: np.ndarray
    m: np.ndarray
    T: float
    dt: float
    box: np.ndarray # [[xmin,xmax],[ymin,ymax]]
    K: int=1
    def __post_init__(self):
        self.t=np.ascontiguousarray(self.t,dtype=np.float64); self.xy=np.ascontiguousarray(self.xy,dtype=np.float64).reshape(-1,2); self.m=np.ascontiguousarray(self.m,dtype=np.int64); self.box=np.asarray(self.box,float)
        if len(self.t)!=len(self.xy) or len(self.t)!=len(self.m): raise ValueError('Inconsistent data lengths')
        if self.T<=0 or self.dt<=0 or self.box.shape!=(2,2) or np.any(self.box[:,1]<=self.box[:,0]): raise ValueError('Invalid domain')
        if not np.isfinite(self.t).all() or not np.isfinite(self.xy).all() or np.any(np.diff(self.t)<=0) or np.any(self.t<=0) or np.any(self.t>=self.T): raise ValueError('Times must be distinct, sorted and strictly inside (0,T). Check timestamp resolution.')
        if np.any(self.xy<self.box[:,0]) or np.any(self.xy>self.box[:,1]) or np.any(self.m<0) or np.any(self.m>=self.K): raise ValueError('Invalid position or mark')
    @property
    def B(self): return int(np.ceil(self.T/self.dt-1e-10))
    def prefix(self,T):
        take=self.t<T
        return Data(self.t[take],self.xy[take],self.m[take],T,self.dt,self.box,self.K)

@dataclass
class Background:
    box: np.ndarray
    centers: np.ndarray
    widths: np.ndarray
    weights: np.ndarray # first component uniform
    def density(self,xy):
        xy=np.asarray(xy).reshape(-1,2); area=np.prod(self.box[:,1]-self.box[:,0]); out=np.full(len(xy),self.weights[0]/area)
        for mu,sd,w in zip(self.centers,self.widths,self.weights[1:]):
            a=(self.box[:,0]-mu)/sd; b=(self.box[:,1]-mu)/sd
            out+=w*np.prod(truncnorm.pdf(xy,a,b,loc=mu,scale=sd),axis=1)
        return out
    def sample(self,rng,n=1):
        k=rng.choice(len(self.weights),size=n,p=self.weights); out=rng.uniform(self.box[:,0],self.box[:,1],(n,2))
        for j,(mu,sd) in enumerate(zip(self.centers,self.widths),1):
            idx=np.flatnonzero(k==j)
            if len(idx): out[idx]=truncnorm.rvs((self.box[:,0]-mu)/sd,(self.box[:,1]-mu)/sd,loc=mu,scale=sd,size=(len(idx),2),random_state=rng)
        return out

@njit(cache=True)
def logcdf(x):
    # erfc is stable until the very far left tail; asymptotic series thereafter.
    if x < -32.:
        z=1/(x*x)
        return -.5*x*x-math.log(-x)-.5*math.log(2*math.pi)+math.log(1-z+3*z*z-15*z**3+105*z**4-945*z**5)
    return math.log(.5*math.erfc(-x/math.sqrt(2)))

@njit(cache=True)
def logdiffcdf(a,b):
    if a>=0: return logdiffcdf(-b,-a)
    lb=logcdf(b); la=logcdf(a)
    if la>=lb: return -np.inf
    return lb+math.log(-math.expm1(la-lb))

@njit(cache=True)
def logadd(a,b):
    if a==-np.inf: return b
    if b==-np.inf: return a
    if a>b: return a+math.log1p(math.exp(b-a))
    return b+math.log1p(math.exp(a-b))

@njit(cache=True)
def clocks(t,z,dt,cf,df):
    B=len(z); ap=np.zeros(B+1); cp=np.zeros(B+1)
    for b in range(B):
        ap[b+1]=ap[b]+dt*cf[z[b]]; cp[b+1]=cp[b]+dt*df[z[b]]
    A=np.empty(len(t)); C=np.empty(len(t))
    for i in range(len(t)):
        b=min(int(t[i]/dt),B-1); rem=t[i]-b*dt
        A[i]=ap[b]+rem*cf[z[b]]; C[i]=cp[b]+rem*df[z[b]]
    return A,C

@njit(cache=True)
def kernel_log(x,y,px,py,a,c,v,q1,q2,r1,r2,box):
    m1=px+v*c; m2=py; s1=math.sqrt(r1*r1+q1*max(0.,a)); s2=math.sqrt(r2*r2+q2*max(0.,a))
    g1=logdiffcdf((box[0,0]-m1)/s1,(box[0,1]-m1)/s1)
    g2=logdiffcdf((box[1,0]-m2)/s2,(box[1,1]-m2)/s2)
    return -math.log(2*math.pi*s1*s2)-.5*((x-m1)/s1)**2-.5*((y-m2)/s2)**2-g1-g2

@njit(cache=True)
def loglik_core(t,xy,m,T,dt,box,f,z,cf,df,v,q1,q2,r1,r2,beta,bg,h,alpha,pi,L):
    """Exact conditional log likelihood of chosen memory model; normalized rectangle.
    Compensator integrates each exponential over intersected grid bins analytically.
    """
    A,C=clocks(t,z,dt,cf,df); B=len(z); n=len(t); comp=0.; ll=0.; left=0
    for b in range(B): comp+=bg[z[b]]*max(0.,min(T,(b+1)*dt)-b*dt)
    for i in range(n):
        r=z[min(int(t[i]/dt),B-1)]; loglam=math.log(bg[r])+math.log(f[i])
        while left<i and t[i]-t[left]>L: left+=1
        for j in range(left,i):
            if alpha[m[j]]<=0: continue
            g=kernel_log(xy[i,0],xy[i,1],xy[j,0],xy[j,1],A[i]-A[j],C[i]-C[j],v,q1,q2,r1,r2,box)
            loglam=logadd(loglam,math.log(h[r]*alpha[m[j]]*beta)-beta*(t[i]-t[j])+g)
        ll+=loglam+math.log(pi[m[i]])
        if alpha[m[i]]<=0: continue
        stop=min(T,t[i]+L); b=min(int(t[i]/dt),B-1)
        while b<B and b*dt<stop:
            lo=max(t[i],b*dt); hi=min(stop,(b+1)*dt)
            if hi>lo: comp+=alpha[m[i]]*h[z[b]]*math.exp(-beta*(lo-t[i]))*(-math.expm1(-beta*(hi-lo)))
            b+=1
    return ll-comp

@dataclass
class Params:
    v: float; q1: float; q2: float; r1: float; r2: float; beta: float
    bg: np.ndarray; h: np.ndarray; alpha: np.ndarray; pi: np.ndarray
    nu: np.ndarray; prob: np.ndarray; P: np.ndarray

def ll(data,bgd,z,p,model,L=np.inf,f=None):
    if f is None: f=bgd.density(data.xy)
    cf,df=factors(model)
    # fixed-state comparators use state index 1, not pause index 0
    return float(loglik_core(data.t,data.xy,data.m,data.T,data.dt,data.box,f,z,cf,df,p.v,p.q1,p.q2,p.r1,p.r2,p.beta,p.bg,p.h,p.alpha,p.pi,L))

def allowed(model): return np.array([1]) if states(model)==1 else np.arange(states(model))

def transition_matrix(model):
    a=allowed(model); P=np.zeros((3,3))
    for r in a:
        if len(a)>1: P[r,a[a!=r]]=1/(len(a)-1)
    return P

def path_sample(rng,B,p,model,initial=None):
    a=allowed(model)
    if len(a)==1: return np.full(B,a[0],dtype=np.int64)
    z=[]; r=int(rng.choice(a) if initial is None else initial)
    while len(z)<B:
        d=1+rng.negative_binomial(p.nu[r],p.prob[r]); z.extend([r]*min(d,B-len(z)))
        if len(z)<B: r=int(rng.choice(3,p=p.P[r]))
    return np.asarray(z,dtype=np.int64)

def runs(z):
    cuts=np.r_[0,np.flatnonzero(z[1:]!=z[:-1])+1,len(z)]
    return [(int(z[a]),int(b-a)) for a,b in zip(cuts[:-1],cuts[1:])]

def path_logprior(z,p,model):
    a=allowed(model)
    if len(a)==1: return 0. if np.all(z==a[0]) else -np.inf
    rr=runs(z); out=-math.log(len(a))
    for k,(r,d) in enumerate(rr):
        if r not in a: return -np.inf
        if k<len(rr)-1: out+=nbinom.logpmf(d-1,p.nu[r],p.prob[r])+math.log(p.P[r,rr[k+1][0]])
        else: out+=nbinom.logsf(d-2,p.nu[r],p.prob[r]) # P(D >= d)
    return float(out)

def extend_path(rng,z,B,p,model):
    if B<=len(z): return z[:B].copy()
    if states(model)==1: return np.full(B,1,dtype=np.int64)
    r,d=runs(z)[-1]; out=list(z)
    # sample duration conditional on surviving d observed bins, via inverse survival.
    sf=nbinom.sf(d-2,p.nu[r],p.prob[r]);
    if not sf>0: raise RuntimeError('Numerically zero duration survival')
    target=np.log(sf)+np.log(rng.uniform())
    low=d-2; high=max(d,2)
    while nbinom.logsf(high,p.nu[r],p.prob[r])>target: high=2*high+2
    while high-low>1:
        mid=(high+low)//2
        if nbinom.logsf(mid,p.nu[r],p.prob[r])>target: low=mid
        else: high=mid
    total=high+1
    out.extend([r]*min(max(0,total-d),B-len(out)))
    while len(out)<B:
        r=int(rng.choice(3,p=p.P[r])); dur=1+rng.negative_binomial(p.nu[r],p.prob[r]); out.extend([r]*min(dur,B-len(out)))
    return np.array(out,dtype=np.int64)

class Parameterization:
    """Unconstrained coordinates with lognormal and beta priors on physical scales.
    Fixed background basis/weights, initial state uniform; optional duration/transition learning.
    Mark probabilities integrated separately (Dirichlet conjugate, sampled for forecasting).
    """
    def __init__(self,model,K,dt,learn_duration=True):
        self.model=model; self.K=K; self.dt=dt; self.a=allowed(model); self.learn=learn_duration and len(self.a)>1
        self.spec=[]
        def ln(name,med,sd,lo,hi): self.spec.append((name,'log',math.log(med),sd,math.log(lo),math.log(hi)))
        def be(name,a,b): self.spec.append((name,'beta',a,b,-15.,15.))
        if model!='M0':
            if model!='M1':
                ln('v',10,1.2,.1,300); ln('sqrt_q1',6,1,.2,100); ln('sqrt_q2',6,1,.2,100)
            ln('r1',3,.8,.3,30); ln('r2',3,.8,.3,30); ln('beta',2,1,.05,100)
            for k in range(K): be(f'alpha{k}',2,2)
            for r in self.a:
                if r!=1: be(f'h{r}',2,2)
        for r in self.a: ln(f'b{r}',10,1.5,.01,2000)
        if self.learn:
            for r in self.a:
                ln(f'nu{r}',2,.7,.2,20); be(f'p{r}',2,5)
                if len(self.a)==3: be(f'P{r}',1,1)
        self.names=[x[0] for x in self.spec]
    def initial(self,rng,rate):
        x=np.array([s[2] if s[1]=='log' else logit(s[2]/(s[2]+s[3])) for s in self.spec]);
        for i,n in enumerate(self.names):
            if n.startswith('b') and n!='beta': x[i]=np.log(np.clip(rate*.6,.02,1500))
        x+=rng.normal(0,.25,len(x)); return np.clip(x,[s[4]+1e-5 for s in self.spec],[s[5]-1e-5 for s in self.spec])
    def prior(self,x):
        val=0.
        for u,(_,typ,a,b,lo,hi) in zip(x,self.spec):
            if not lo<u<hi: return -np.inf
            if typ=='log': val-=.5*((u-a)/b)**2
            else: val+=a*(-np.logaddexp(0,-u))+b*(-np.logaddexp(0,u)) # beta plus logit Jacobian
        return float(val)
    def decode(self,x,pi):
        d={s[0]: (np.exp(u) if s[1]=='log' else expit(u)) for u,s in zip(x,self.spec)}
        model=self.model; bg=np.ones(3); h=np.ones(3); nu=np.full(3,2.); prob=np.array([.4,.2,.2]); P=transition_matrix(model)
        for r in self.a:
            bg[r]=d[f'b{r}']; h[r]=d.get(f'h{r}',1.)
            if self.learn:
                nu[r]=d[f'nu{r}']; prob[r]=d[f'p{r}']
                if len(self.a)==3:
                    oth=self.a[self.a!=r]; P[r,oth[0]]=d[f'P{r}']; P[r,oth[1]]=1-d[f'P{r}']
            # Recalibrate the NB mean duration in physical days relative to 6-hour grid.
            mean_extra=nu[r]*(1-prob[r])/prob[r]; target=max(.05,(1+mean_extra)*.25/self.dt-1); prob[r]=nu[r]/(nu[r]+target)
        alpha=np.array([RHO*d.get(f'alpha{k}',0.) for k in range(self.K)])
        return Params(d.get('v',0.),d.get('sqrt_q1',0.)**2,d.get('sqrt_q2',0.)**2,d.get('r1',3.),d.get('r2',3.),d.get('beta',2.),bg,h,alpha,np.asarray(pi),nu,prob,P)
    def natural(self,x):
        d={s[0]: (np.exp(u) if s[1]=='log' else expit(u)) for u,s in zip(x,self.spec)}
        for k in range(self.K):
            if f'alpha{k}' in d: d[f'alpha{k}']*=RHO
        return d


def fit(data,bgd,model='M5',memory=4.,chains=4,steps=4000,burn=2000,thin=4,seed=123,learn_duration=True,progress=True):
    """Metropolis-within-Gibbs on parameters and entire path, not PMMH.
    Symmetric interval label-swap proposal, plus a whole-path prior proposal.
    Adapt parameter step sizes only in burn-in. All rejected draws are retained.
    """
    if not (0<=burn<steps and thin>=1 and chains>=1 and memory>0): raise ValueError('Invalid sampler settings')
    par=Parameterization(model,data.K,data.dt,learn_duration); f=bgd.density(data.xy); pi=(np.bincount(data.m,minlength=data.K)+1)/(len(data.t)+data.K)
    nsave=len(range(burn,steps,thin)); X=np.empty((chains,nsave,len(par.names))); Z=np.empty((chains,nsave,data.B),np.int8); LP=np.empty((chains,nsave)); stats=[]; start=time.time()
    for ch in range(chains):
        rng=np.random.default_rng(seed+100003*ch); x=par.initial(rng,len(data.t)/data.T); p=par.decode(x,pi); z=path_sample(rng,data.B,p,model); l=ll(data,bgd,z,p,model,memory,f); zp=path_logprior(z,p,model); pp=par.prior(x)
        if not np.isfinite(l+zp+pp): raise RuntimeError('Nonfinite initialization')
        scale=np.full(len(x),.18); att=np.zeros(len(x)); acc=np.zeros(len(x)); npa=npt=nza=nzt=0; save=0
        for it in range(steps):
            # random coordinate or joint random-walk; coordinate update keeps tuning simple
            j=int(rng.integers(len(x))); xp=x.copy(); xp[j]+=rng.normal(0,scale[j]); pprior=par.prior(xp); accept=False
            if np.isfinite(pprior):
                pn=par.decode(xp,pi); zpn=path_logprior(z,pn,model); ln=ll(data,bgd,z,pn,model,memory,f)
                if np.log(rng.random())<ln+zpn+pprior-l-zp-pp:
                    x,p,zp,pp,l=xp,pn,zpn,pprior,ln; accept=True
            if it<burn:
                att[j]+=1; acc[j]+=accept
                if att[j]>=20:
                    scale[j]=np.clip(scale[j]*np.exp(.5*(acc[j]/att[j]-.35)),.015,1.5); att[j]=acc[j]=0
            else: npa+=accept; npt+=1
            if states(model)>1:
                # Independent prior proposal (ratio = likelihood ratio); or symmetric swap.
                if rng.random()<.06:
                    zn=path_sample(rng,data.B,p,model); zpn=path_logprior(zn,p,model); prior_proposal=True
                else:
                    zn=z.copy(); width=int(rng.choice([1,2,4,8,16,32])); i=int(rng.integers(data.B)); j2=min(data.B,i+width)
                    a,b=rng.choice(par.a,2,replace=False); seg=zn[i:j2].copy(); zn[i:j2]=np.where(seg==a,b,np.where(seg==b,a,seg)); zpn=path_logprior(zn,p,model); prior_proposal=False
                changed=not np.array_equal(zn,z); za=False
                if changed and np.isfinite(zpn):
                    ln=ll(data,bgd,zn,p,model,memory,f); ratio=ln-l+(0. if prior_proposal else zpn-zp)
                    if np.log(rng.random())<ratio: z,zp,l=zn,zpn,ln; za=True
                if it>=burn and changed: nza+=za; nzt+=1
            if it>=burn and (it-burn)%thin==0: X[ch,save]=x; Z[ch,save]=z; LP[ch,save]=l; save+=1
        stats.append(dict(chain=ch+1,parameter_acceptance=npa/max(1,npt),path_acceptance=nza/nzt if nzt else np.nan,path_attempts=nzt,loglik_last=l))
        if progress: print(f'{model}: chain {ch+1}/{chains} | parameter acceptance {stats[-1]["parameter_acceptance"]:.2f} | path {stats[-1]["path_acceptance"]:.2f}',flush=True)
    return dict(X=X,Z=Z,loglik=LP,stats=pd.DataFrame(stats),par=par,pi=pi,mark_counts=np.bincount(data.m,minlength=data.K),memory=memory,model=model,seconds=time.time()-start)


def reference_params(K=1,dt=.25):
    return Params(10.,36.,4.,3.,2.,2.,np.array([6.,12.,12.]),np.array([.65,1.,.85]),np.array([.55]) if K==1 else np.array([.4,.7]),np.ones(1) if K==1 else np.array([.7,.3]),np.full(3,2.),np.array([.4,.2,.2]),transition_matrix('M5'))

def simulate(rng,T,dt,box,bgd,z,p,model='M5',memory=np.inf,origin=0.,history=None,max_events=100000):
    """Exact Poisson cluster simulation conditional on path. Forecasts condition on
    observed history: each historical event contributes only future offspring.
    No time grid approximation other than state bins. Guard raises, never truncates data.
    """
    events=[]; cf,df=factors(model); B=len(z)
    def state(t): return int(z[min(int(t/dt),B-1)])
    for b in range(B):
        lo=max(origin,b*dt); hi=min(T,(b+1)*dt)
        if hi<=lo: continue
        n=rng.poisson(p.bg[z[b]]*(hi-lo)); tt=rng.uniform(lo,hi,n); loc=bgd.sample(rng,n); marks=rng.choice(len(p.pi),size=n,p=p.pi)
        events.extend((float(t),xy,int(k)) for t,xy,k in zip(tt,loc,marks))
    old=[] if history is None else [(float(t),xy,int(k)) for t,xy,k in zip(history.t,history.xy,history.m) if t<origin and t+memory>origin]
    # process historical parents exactly once, then new immigrants and accepted descendants
    queue=old+events.copy(); idx=0
    while idx<len(queue):
        t,xy,k=queue[idx]; idx+=1; lo=max(0.,origin-t); hi=min(T-t,memory)
        if hi<=lo or p.alpha[k]<=0: continue
        a=math.exp(-p.beta*lo); b=math.exp(-p.beta*hi); mass=a-b
        if mass<=0: continue
        nc=rng.poisson(p.alpha[k]*mass)
        for lag in -np.log(rng.uniform(b,a,nc))/p.beta:
            child=t+lag
            if not origin<child<T or rng.random()>p.h[state(child)]: continue
            A,C=clocks(np.array([t,child]),z,dt,cf,df); mu=xy+np.array([p.v*(C[1]-C[0]),0.]); sd=np.sqrt(np.array([p.r1**2,p.r2**2])+max(0.,A[1]-A[0])*np.array([p.q1,p.q2]))
            loc=truncnorm.rvs((box[:,0]-mu)/sd,(box[:,1]-mu)/sd,loc=mu,scale=sd,random_state=rng); mark=int(rng.choice(len(p.pi),p=p.pi)); ev=(child,loc,mark); events.append(ev); queue.append(ev)
            if len(events)>max_events: raise RuntimeError('Simulation event guard exceeded; do not silently truncate')
    events.sort(key=lambda r:r[0]); return Data(np.array([r[0] for r in events]),np.array([r[1] for r in events]).reshape(-1,2),np.array([r[2] for r in events],int),T,dt,box,len(p.pi))


def scenario(s,seed,T=30.,dt=.25):
    if T!=30 or dt!=.25: raise ValueError('Prespecified scenarios use T=30, dt=.25')
    rng=np.random.default_rng(seed); box=np.array([[-150.,150.],[-40.,40.]]); K=2 if s=='S8' else 1; p=reference_params(K,dt); B=int(T/dt); bgd=Background(box,np.array([[0.,0.]]),np.array([[50.,12.]]),np.array([.2,.8])); z=path_sample(rng,B,p,'M5',initial=1); gen='M5'
    if s=='S1': z[:]=1
    if s in ('S2','S3'): z=np.repeat([1,0,1,0,2],[32,8,32,8,40])
    if s=='S3': gen='S3'
    if s=='S5': p.alpha[:]=.1; p.v=1.; p.r1=10.
    if s=='S7': gen='S7'
    data=simulate(rng,T,dt,box,bgd,z,p,gen)
    if s=='S6':
        retention=np.where(((data.t>=8)&(data.t<10))|((data.t>=18)&(data.t<20)),.2,1.); k=rng.random(len(data.t))<retention; data=Data(data.t[k],data.xy[k],data.m[k],T,dt,box,K)
    return data,bgd,z,p,gen

# -------------------- Reporting and predictive verification --------------------
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.17,'figure.dpi':125,'savefig.dpi':300,'pdf.fonttype':42,'font.family':'DejaVu Sans'})
COL=['#AA563B','#246B8E','#6D5295']; MODEL_COL=dict(zip(MODELS,['#999999','#486B82','#578C87','#C28D38','#6D5295','#186E66','#8A5656']))

def savefig(fig,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    for ext in ('.png','.pdf'): fig.savefig(path.with_suffix(ext),bbox_inches='tight',facecolor='white')
    plt.close(fig)

def table(df,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); df.to_csv(path.with_suffix('.csv'),index=False)
    path.with_suffix('.tex').write_text(df.to_latex(index=False,float_format=lambda x:f'{x:.4g}',escape=True),encoding='utf8')
    path.with_suffix('.html').write_text('<meta charset="utf-8"><style>body{font-family:Arial;margin:24px}table{border-collapse:collapse}th{background:#17455a;color:white}td,th{padding:7px 10px;border-bottom:1px solid #ddd;text-align:right}tr:nth-child(even){background:#f1f5f6}</style>'+df.to_html(index=False,float_format=lambda x:f'{x:.4g}'),encoding='utf8')

def summary(fitres,data,out):
    out=Path(out); out.mkdir(parents=True,exist_ok=True); X,Z,par=fitres['X'],fitres['Z'],fitres['par']; names=par.names
    natural={k:np.array([[par.natural(x)[k] for x in chain] for chain in X]) for k in names}
    # Report parameters on physical scale, plus meaningful derived quantities.
    if 'sqrt_q1' in natural:
        natural['q1']=natural['sqrt_q1']**2; natural['q2']=natural['sqrt_q2']**2; natural['q_ratio']=natural['q1']/natural['q2']
    if 'beta' in natural: natural['half_life_days']=np.log(2)/natural['beta']
    if fitres['model'] in ('M4','M5','M5slow'):
        cf,df=factors(fitres['model']); widths=np.minimum(data.dt,data.T-np.arange(data.B)*data.dt)
        natural['A_T']=np.sum(cf[Z]*widths,axis=-1); natural['C_T']=np.sum(df[Z]*widths,axis=-1); natural['signed_km']=natural['C_T']*natural['v']; natural['slow_fraction' if fitres['model']=='M5slow' else 'pause_fraction']=np.sum((Z==0)*widths,axis=-1)/data.T
    def diagnostic(arr):
        if arr.shape[0]<2 or arr.shape[1]<4 or np.ptp(arr)==0: return np.nan,np.nan,np.nan
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            return float(az.rhat(arr,method='rank')),float(az.ess(arr,method='bulk')),float(az.ess(arr,method='tail',prob=(.05,.95)))
    rows=[]
    for k,arr in natural.items():
        rh,eb,et=diagnostic(arr); lo,hi=np.quantile(arr,[.05,.95]); rows.append(dict(parameter=k,mean=arr.mean(),sd=arr.std(ddof=1),q05=lo,q95=hi,r_hat=rh,ess_bulk=eb,ess_tail=et))
    s=pd.DataFrame(rows)
    table(s,out/'posterior_summary'); table(fitres['stats'],out/'sampler_acceptance')
    np.savez_compressed(out/'posterior_draws.npz',X=X,Z=Z,loglik=fitres['loglik'],names=np.array(names),pi=fitres['pi'],mark_counts=fitres['mark_counts'],box=data.box,dt=data.dt,T=data.T,memory=fitres['memory'],model=fitres['model'],learn_duration=par.learn)
    diag=s[['parameter','r_hat','ess_bulk','ess_tail']].copy(); diag['flag']=np.where((diag.r_hat>1.01)|(diag.ess_bulk<400)|(diag.ess_tail<400)|~np.isfinite(diag.r_hat),'REVIEW','OK'); table(diag,out/'diagnostics')
    tracekeys=[k for k in ['v','beta','r1','alpha0','b1','q1','pause_fraction'] if k in natural][:6]
    fig,ax=plt.subplots(len(tracekeys)+1,1,figsize=(8,1.45*(len(tracekeys)+1)),sharex=True,layout='constrained'); ax=np.atleast_1d(ax)
    for a,k in zip(ax,tracekeys):
        for c in natural[k]: a.plot(c,lw=.65,alpha=.8)
        a.set_ylabel(k)
    for c in fitres['loglik']: ax[-1].plot(c,lw=.65)
    ax[-1].set(ylabel='Log likelihood',xlabel='Retained draw'); fig.suptitle(f'{LABELS[fitres["model"]]} | MCMC traces'); savefig(fig,out/'trace')
    if fitres['model'] in ('M4','M5','M5slow'):
        probs=np.stack([(Z==k).mean(axis=(0,1)) for k in range(3)],axis=1); rows=pd.DataFrame({'bin_start_day':np.arange(data.B)*data.dt,'p_pause':probs[:,0],'p_forward':probs[:,1],'p_reverse':probs[:,2]})
        # Chain variation and MCMC diagnostic for indicator, not i.i.d. binomial SE.
        rows['p_pause_chain_range']=np.ptp((Z==0).mean(axis=1),axis=0)
        ds=np.array([diagnostic((Z[:,:,b]==0).astype(float)) for b in range(data.B)]); rhat=ds[:,0]; ess=ds[:,1]
        rows['pause_rhat']=rhat; rows['pause_ess_bulk']=ess; rows['pause_mcse_approx']=np.sqrt(probs[:,0]*(1-probs[:,0])/np.maximum(1,ess)); table(rows.rename(columns={k:k.replace('pause','slow') for k in rows.columns if 'pause' in k}) if fitres['model']=='M5slow' else rows,out/'state_probabilities')
        plot_states(data,fitres,out/'transport')
    return s,natural

def plot_states(data,res,path,truth=None):
    z=res['Z']; cf,df=factors(res['model']); widths=np.minimum(data.dt,data.T-np.arange(data.B)*data.dt); grid=np.r_[0,np.cumsum(widths)]; pr=np.stack([(z==r).mean((0,1)) for r in range(3)]); A=np.concatenate([np.zeros((*z.shape[:2],1)),np.cumsum(cf[z]*widths,axis=-1)],axis=-1); C=np.concatenate([np.zeros((*z.shape[:2],1)),np.cumsum(df[z]*widths,axis=-1)],axis=-1)
    fig,ax=plt.subplots(3,1,figsize=(9,7),sharex=True,layout='constrained',gridspec_kw={'height_ratios':[2,1,1.5]}); ax[0].scatter(data.t,data.xy[:,0],s=4,alpha=.45,color='#22566F',rasterized=True); ax[0].set(ylabel='Along-strike (km)',title='Detected events and posterior transport state')
    for r in range(3): ax[1].stairs(pr[r],grid,color=COL[r],label=(['Slow','Forward','Reverse'] if res['model']=='M5slow' else ['Pause','Forward','Reverse'])[r],lw=1.3)
    if truth is not None: ax[1].stairs((truth==0).astype(float),grid,color='black',linestyle='--',label='True pause',lw=1)
    ax[1].set(ylabel='Probability',ylim=(-.03,1.03)); ax[1].legend(loc='upper right',ncol=4,fontsize=8)
    for arr,color,label in [(A,'#246B8E','Active time'),(C,'#AA563B','Signed time')]:
        q=np.quantile(arr,[.05,.5,.95],axis=(0,1)); ax[2].fill_between(grid,q[0],q[2],color=color,alpha=.17); ax[2].plot(grid,q[1],color=color,label=label)
    ax[2].plot(grid,grid,ls=':',color='grey',label='Chronological time'); ax[2].set(xlabel='Elapsed time (days)',ylabel='Clock (days)'); ax[2].legend(ncol=3,fontsize=8); savefig(fig,path)

def crps(draws,y):
    x=np.sort(np.asarray(draws,float),axis=0); n=len(x); coeff=(2*np.arange(1,n+1)-n-1).reshape((n,)+(1,)*(x.ndim-1)); return np.mean(np.abs(x-y),axis=0)-np.sum(coeff*x,axis=0)/(n*n)

def forecast(data,bgd,res,origin,horizons=(.25,1.,3.),ndraw=60,seed=987,out=None):
    """Posterior predictive continuation, with observed pre-origin history and
    censored run survival. Configuration score via log mean exp of conditional
    likelihood ratios, evaluated using ONLY the training posterior.
    """
    assert abs(origin/data.dt-round(origin/data.dt))<1e-8,'Origin must be on state grid'
    hist=data.prefix(origin); maxT=origin+max(horizons); assert maxT<=data.T+1e-8
    assert res['Z'].shape[-1]==hist.B
    rng=np.random.default_rng(seed); flatX=res['X'].reshape(-1,res['X'].shape[-1]); flatZ=res['Z'].reshape(-1,hist.B); par=res['par']; model=res['model']; mem=res['memory']; counts=np.zeros((ndraw,len(horizons)),int); cells=np.zeros((ndraw,len(horizons),16),int); logs=np.zeros((ndraw,len(horizons))); xedges=np.linspace(*data.box[0],5); yedges=np.linspace(*data.box[1],5)
    def hist2(xy): return np.histogram2d(xy[:,0],xy[:,1],bins=[xedges,yedges])[0].ravel().astype(int)
    f_hist=bgd.density(hist.xy); tests=[data.prefix(origin+h) for h in horizons]; f_test=[bgd.density(d.xy) for d in tests]
    for k,i in enumerate(rng.integers(len(flatX),size=ndraw)):
        pi=rng.dirichlet(res['mark_counts']+1); p=par.decode(flatX[i],pi); z=extend_path(rng,flatZ[i],int(np.ceil(maxT/data.dt-1e-10)),p,model); sim=simulate(rng,maxT,data.dt,data.box,bgd,z,p,model,mem,origin,hist)
        base=ll(hist,bgd,flatZ[i],p,model,mem,f_hist)
        for j,h in enumerate(horizons):
            ksel=sim.t<origin+h; counts[k,j]=ksel.sum(); cells[k,j]=hist2(sim.xy[ksel]); d=tests[j]; logs[k,j]=ll(d,bgd,z[:d.B],p,model,mem,f_test[j])-base
    rows=[]
    for j,h in enumerate(horizons):
        obs=tests[j].xy[tests[j].t>=origin]; y=len(obs); yc=hist2(obs); lo,hi=np.quantile(counts[:,j],[.05,.95]); lp=logsumexp(logs[:,j])-np.log(ndraw); lw=np.exp(logs[:,j]-logsumexp(logs[:,j]));
        rows.append(dict(model=model,origin_day=origin,horizon_day=h,observed=y,pred_mean=counts[:,j].mean(),q05=lo,q95=hi,coverage90=int(lo<=y<=hi),count_CRPS=float(crps(counts[:,j],y)),grid_CRPS=float(crps(cells[:,j],yc).mean()),configuration_log_score=lp,configuration_weight_ESS=1/np.sum(lw**2),predictive_draws=ndraw))
    tab=pd.DataFrame(rows)
    if out is not None:
        out=Path(out); out.mkdir(parents=True,exist_ok=True); table(tab,out/'forecast_scores'); np.savez_compressed(out/'predictive_draws.npz',counts=counts,grid_counts=cells,conditional_log_density=logs,horizons=horizons,origin=origin,xedges=xedges,yedges=yedges)
        fig,ax=plt.subplots(1,2,figsize=(9,3.5),layout='constrained'); xx=np.arange(len(horizons)); ax[0].errorbar(xx,tab.pred_mean,yerr=[tab.pred_mean-np.minimum(tab.q05,tab.pred_mean),np.maximum(tab.q95,tab.pred_mean)-tab.pred_mean],fmt='o',capsize=4,color='#246B8E',label='Predictive mean / 90% interval'); ax[0].scatter(xx,tab.observed,marker='x',s=50,color='#AA563B',label='Observed'); ax[0].set(xticks=xx,xticklabels=[f'{h*24:g} h' for h in horizons],ylabel='Count',xlabel='Horizon',title='Held-out event counts'); ax[0].legend(fontsize=8)
        obs=hist2(tests[-1].xy[tests[-1].t>=origin]); pred=cells[:,-1].mean(0); im=ax[1].imshow((pred-obs).reshape(4,4).T,origin='lower',cmap='RdBu_r',vmin=-max(1,np.max(np.abs(pred-obs))),vmax=max(1,np.max(np.abs(pred-obs))),extent=[*data.box[0],*data.box[1]],aspect='auto'); ax[1].set(title='Forecast residual: longest horizon',xlabel='Along-strike (km)',ylabel='Cross-strike (km)'); fig.colorbar(im,ax=ax[1],label='Predicted − observed count'); savefig(fig,out/'forecast')
    return tab


def sim_metrics(data,res,z,p,s,summary_df,out):
    rows=[]; model=res['model']; correct=s in ('S1','S2','S4','S5','S8'); truth={'v':p.v,'q1':p.q1,'q2':p.q2,'r1':p.r1,'r2':p.r2,'beta':p.beta,**{f'alpha{k}':v for k,v in enumerate(p.alpha)}}
    if model=='M5' and correct:
        truth.update(A_T=np.sum((z!=0)*data.dt),C_T=np.sum(np.where(z==1,1,np.where(z==2,-1,0))*data.dt)); truth['signed_km']=p.v*truth['C_T']; truth['pause_fraction']=np.mean(z==0)
        for k,v in truth.items():
            d=summary_df[summary_df.parameter==k]
            if len(d):
                a=d.iloc[0]; rows.append(dict(parameter=k,truth=v,estimate=a['mean'],error=a['mean']-v,covered90=int(a.q05<=v<=a.q95)))
    metric={'model':model,'scenario':s,'n_train':len(data.t),'seconds':res['seconds'],'max_rhat':summary_df.r_hat.max(),'min_ess_bulk':summary_df.ess_bulk.min(),'diagnostic_review':bool(((summary_df.r_hat>1.01)|(summary_df.ess_bulk<400)|~np.isfinite(summary_df.r_hat)).any())}
    if model in ('M4','M5'):
        pp=np.stack([(res['Z']==r).mean((0,1)) for r in range(3)],axis=1); pred=pp[:,0]>.5; truepause=z==0 if s not in ('S3','S7') else np.zeros(len(z),bool); tp=(pred&truepause).sum(); metric.update(pause_prob_mean=pp[:,0].mean(),false_pause_rate=float((pred&~truepause).sum()/max(1,(~truepause).sum())),pause_precision=tp/pred.sum() if pred.sum() else np.nan,pause_recall=tp/truepause.sum() if truepause.sum() else np.nan,pause_brier=np.mean((pp[:,0]-truepause)**2))
        if s not in ('S3','S7'):
            metric['multiclass_brier']=np.mean(np.sum((pp-np.eye(3)[z])**2,axis=1)); metric['active_clock_abs_error']=abs(np.sum((1-pp[:,0])*data.dt)-np.sum((z!=0)*data.dt)) if model!='M5slow' else np.nan
        # Per-bin truth + probabilities for calibration and boundary audit.
        table(pd.DataFrame({'time_day':np.arange(data.B)*data.dt,'true_generator_state':z,'true_pause':truepause.astype(int),'p_pause':pp[:,0],'p_forward':pp[:,1],'p_reverse':pp[:,2]}),Path(out)/'truth_state_comparison')
        if s not in ('S3','S7'):
            true_b=list(np.flatnonzero(truepause[1:]!=truepause[:-1])+1); est_b=list(np.flatnonzero(pred[1:]!=pred[:-1])+1); unmatched=set(est_b); errs=[]
            for b in true_b:
                near=[e for e in unmatched if abs(e-b)<=1]
                if near: e=min(near,key=lambda x:abs(x-b)); unmatched.remove(e); errs.append(abs(e-b)*data.dt)
            metric.update(boundary_matched=len(errs),boundary_unmatched_true=len(true_b)-len(errs),boundary_unmatched_est=len(unmatched),boundary_mae_days=np.mean(errs) if errs else np.nan)
    if rows: table(pd.DataFrame(rows),Path(out)/'parameter_recovery')
    return metric,rows


def checks():
    """Numerical checks independent of the MCMC: density integration, exact
    compensator against quadrature, truncation identity, S2 clocks, simulator."""
    from scipy.integrate import quad
    from scipy.special import log_ndtr
    from scipy.stats import norm
    from scipy.integrate import dblquad
    box=np.array([[-8.,8.],[-5.,5.]]); p=reference_params(); bgd=Background(box,np.empty((0,2)),np.empty((0,2)),np.ones(1)); z=np.array([1,0,2,1]); dt=.25
    for a,b in [(-90,-89.8),(8,9),(-1,1),(-.1,.1),(-10,-9.9)]:
        if a>=0: a,b=-b,-a
        ref=log_ndtr(b)+np.log(-np.expm1(log_ndtr(a)-log_ndtr(b))); assert abs(logdiffcdf(a,b)-ref)<1e-7
    fun=lambda y,x: np.exp(kernel_log(x,y,7.,4.,.7,.7,10.,36.,4.,3.,2.,box)); mass=dblquad(fun,*box[0],lambda x:box[1,0],lambda x:box[1,1])[0]; assert abs(mass-1)<1e-7
    data=Data(np.array([.1,.4,.8]),np.array([[0.,0.],[1.,0.],[2.,1.]]),np.zeros(3,int),1.,dt,box); A,C=clocks(data.t,z,dt,*factors('M5')); f=bgd.density(data.xy)
    for L in (.3,np.inf):
        # Explicit event intensity and quadrature split at every state/parent/expiry boundary.
        logev=0.
        for i,t in enumerate(data.t):
            r=z[int(t/dt)]; val=p.bg[r]*f[i]
            for j in range(i):
                if t-data.t[j]<=L: val+=p.h[r]*p.alpha[0]*p.beta*np.exp(-p.beta*(t-data.t[j])+kernel_log(*data.xy[i],*data.xy[j],A[i]-A[j],C[i]-C[j],p.v,p.q1,p.q2,p.r1,p.r2,box))
            logev+=np.log(val)
        def rate(t):
            r=z[min(int(t/dt),3)]; lag=t-data.t; return p.bg[r]+p.h[r]*np.sum(p.alpha[0]*p.beta*np.exp(-p.beta*lag[(lag>0)&(lag<=L)]))
        knots=np.unique(np.r_[0,1,np.arange(4)*dt,data.t,(data.t+L)[data.t+L<1]]); comp=sum(quad(rate,a,b,epsabs=1e-10)[0] for a,b in zip(knots[:-1],knots[1:])); assert abs(ll(data,bgd,z,p,'M5',L)-(logev-comp))<1e-8
    assert abs(ll(data,bgd,z,p,'M5',2)-ll(data,bgd,z,p,'M5',np.inf))<1e-10
    zs=np.repeat([1,0,1,0,2],[32,8,32,8,40]); a,c=clocks(np.array([30.]),zs,.25,*factors('M5')); assert np.allclose([a[0],c[0]],[26,6]); assert np.isclose(crps([0,2],1),.5)
    # Proper final-run survival, a duration-one path has survival 1.
    assert np.isclose(path_logprior(np.array([1]),p,'M5'),-np.log(3))
    for i in range(20):
        ext=extend_path(np.random.default_rng(i),zs,130,p,'M5'); assert np.array_equal(ext[:120],zs) and len(ext)==130
    print('PASS: stable CDF, boundary normalization, quadrature likelihood, memory, S2 clocks, CRPS and path continuation')
    return {'kernel_integral':mass,'S2_active_days':float(a[0]),'S2_signed_days':float(c[0]),'status':'PASS'}


In [ ]:
%%writefile workflow.py
"""Experiment orchestration, catalogue ingestion, reproducible result bundles."""
from dual_clock import *
from datetime import datetime, timezone
import traceback, shutil, platform, importlib.metadata
from pyproj import CRS, Transformer

# Pilot is deliberately NOT publication-grade. Change profile or explicit settings.
PROFILES={
 'SMOKE':dict(chains=2,steps=40,burn=20,thin=2,learn_duration=False,predictive_draws=4,reps=1,models=['M1','M3','M5'],scenarios=['S2'],origins=[25.],memory=2.,real_windows_per_year=1),
 'PILOT':dict(chains=2,steps=2000,burn=1000,thin=4,learn_duration=True,predictive_draws=40,reps=1,models=list(MODELS),scenarios=[f'S{i}' for i in range(1,9)],origins=[25.],memory=3.,real_windows_per_year=1),
 'RESEARCH':dict(chains=4,steps=16000,burn=8000,thin=8,learn_duration=True,predictive_draws=300,reps=20,models=list(MODELS),scenarios=[f'S{i}' for i in range(1,9)],origins=[25.,27.],memory=4.,real_windows_per_year=12),
}

def config(profile='SMOKE',**kwargs):
    c=PROFILES[profile].copy(); c.update(kwargs); c['profile']=profile; return c

def write_json(path,obj):
    Path(path).parent.mkdir(parents=True,exist_ok=True); Path(path).write_text(json.dumps(obj,indent=2,default=str),encoding='utf8')

def init_run(root,c):
    root=Path(root); root.mkdir(parents=True,exist_ok=True); write_json(root/'configuration.json',c)
    versions={k:importlib.metadata.version(k) for k in ['numpy','scipy','pandas','numba','arviz','matplotlib','pyproj']}; write_json(root/'environment.json',dict(python=platform.python_version(),packages=versions,created_utc=datetime.now(timezone.utc).isoformat())); write_json(root/'numerical_checks.json',checks())
    for name in ['dual_clock.py','workflow.py']:
        f=Path(__file__).with_name(name)
        if f.exists(): shutil.copyfile(f,root/name)
    return root

def save_data(d,bg,out):
    out=Path(out); out.mkdir(parents=True,exist_ok=True); pd.DataFrame(dict(time_day=d.t,x_km=d.xy[:,0],y_km=d.xy[:,1],mark=d.m)).to_csv(out/'events.csv',index=False)
    np.savez_compressed(out/'data_and_background.npz',t=d.t,xy=d.xy,m=d.m,T=d.T,dt=d.dt,box=d.box,K=d.K,bg_centers=bg.centers,bg_widths=bg.widths,bg_weights=bg.weights)

def fit_one(d,bg,model,c,seed,out):
    # Explicit cost guard, not a random event cap. Save status and shorten window / region if exceeded.
    n=len(d.t); pairs=int(np.sum(np.arange(n)-np.searchsorted(d.t,d.t-c['memory'],side='left')))
    if pairs>c.get('max_pairs',2_000_000): raise RuntimeError(f'{pairs:,} candidate pairs per likelihood; increase max_pairs deliberately or use a prespecified smaller observation domain/window. No events were subsampled.')
    print(f'  {model}: n={n:,}, pair evaluations/likelihood={pairs:,}',flush=True)
    fitres=fit(d,bg,model,memory=c['memory'],chains=c['chains'],steps=c['steps'],burn=c['burn'],thin=c['thin'],learn_duration=c['learn_duration'],seed=seed)
    ss,nat=summary(fitres,d,out)
    # Bounds from manuscript; they are conditional model bounds, not data-specific errors.
    vals=[]
    for chain in fitres['X']:
        for x in chain:
            p=fitres['par'].decode(x,fitres['pi']); vals.append(np.exp(-p.beta*c['memory']))
    table(pd.DataFrame({'memory_days':[c['memory']],'mean_exp_minus_beta_L':[np.mean(vals)],'q95_exp_minus_beta_L':[np.quantile(vals,.95)],'note':['Increase memory and refit if remaining exponential mass is material.']}),Path(out)/'memory_diagnostic')
    return fitres,ss

def summarize_scores(rows,root,group_col):
    if not rows: return
    scores=pd.DataFrame(rows); table(scores,root/'all_forecast_scores')
    # Aggregate at independent replicate/window level, not event level. Multiple origins within unit are averaged.
    metrics=['count_CRPS','grid_CRPS','configuration_log_score','coverage90']
    unit=scores.groupby([group_col,'unit','model','horizon_day'],as_index=False)[metrics].mean()
    agg=unit.groupby([group_col,'model','horizon_day'])[metrics].agg(['mean','sem','count']); agg.columns=['_'.join(x) for x in agg.columns]; table(agg.reset_index(),root/'forecast_summary')
    # Paired M5 minus comparator; unit-level SEM descriptive for real serially dependent windows.
    paired=[]
    for (group,h),d in unit.groupby([group_col,'horizon_day']):
        for model in sorted(set(d.model)-{'M5'}):
            a=d[d.model=='M5'].set_index('unit'); b=d[d.model==model].set_index('unit'); common=a.index.intersection(b.index)
            for metric in metrics:
                x=(a.loc[common,metric]-b.loc[common,metric]).to_numpy(); paired.append({group_col:group,'horizon_day':h,'comparator':model,'metric':metric,'paired_n':len(x),'M5_minus_comparator':np.mean(x) if len(x) else np.nan,'unit_SE':np.std(x,ddof=1)/np.sqrt(len(x)) if len(x)>1 else np.nan})
    if paired: table(pd.DataFrame(paired),root/'paired_comparisons')
    # 24-hour comparison, separate panels for scenario or period, no arbitrary pooling across scenarios.
    a=unit[unit.horizon_day==1.]; groups=list(a[group_col].unique())
    if groups:
        fig,axes=plt.subplots(len(groups),1,figsize=(8,max(3,2.4*len(groups))),layout='constrained',squeeze=False)
        for ax,g in zip(axes[:,0],groups):
            v=a[a[group_col]==g].groupby('model').count_CRPS.agg(['mean','sem']).reindex([m for m in MODELS if m in set(a[a[group_col]==g].model)]); ax.barh(range(len(v)),v['mean'],xerr=v['sem'].fillna(0),color=[MODEL_COL[m] for m in v.index],alpha=.9); ax.set(yticks=range(len(v)),yticklabels=v.index,title=str(g),xlabel='24-hour count CRPS (lower is better)'); ax.invert_yaxis()
        fig.suptitle('Forecast comparison | error bars: unit-level SEM when available'); savefig(fig,root/'forecast_comparison')

def run_simulation(root,c,seed=20261125,rep_start=0):
    root=Path(root)/'simulation'; root.mkdir(parents=True,exist_ok=True); metrics=[]; recovery=[]; forecasts=[]; failures=[]
    for s in c['scenarios']:
        for rep in range(rep_start,rep_start+c['reps']):
            run=root/s/f'rep_{rep+1:04d}'; run.mkdir(parents=True,exist_ok=True); rseed=seed+100000*int(s[1:])+rep*101; print(f'\n{s} / replicate {rep+1}',flush=True)
            try:
                d,bg,z,p,gen=scenario(s,rseed); save_data(d,bg,run); write_json(run/'truth.json',{'scenario':s,'seed':rseed,'generator':gen,'state_path':z.tolist(),'parameters':vars(p)|{'bg':p.bg.tolist(),'h':p.h.tolist(),'alpha':p.alpha.tolist(),'pi':p.pi.tolist(),'nu':p.nu.tolist(),'prob':p.prob.tolist(),'P':p.P.tolist()}})
            except Exception as e:
                failures.append(dict(scenario=s,rep=rep+1,model='generator',origin=np.nan,error=repr(e))); continue
            for origin in c['origins']:
                train=d.prefix(origin)
                for mi,model in enumerate(c['models']):
                    out=run/f'origin_{origin:g}'/model; out.mkdir(parents=True,exist_ok=True)
                    try:
                        f,ss=fit_one(train,bg,model,c,rseed+1000*mi+int(origin*100),out); mm,rr=sim_metrics(train,f,z[:train.B],p,s,ss,out); mm.update(unit=rep+1,origin_day=origin); metrics.append(mm); recovery.extend([r|{'scenario':s,'unit':rep+1,'origin_day':origin} for r in rr]);
                        if model in ('M4','M5','M5slow') and s not in ('S3','S7'): plot_states(train,f,out/'transport_vs_truth',z[:train.B])
                        fc=forecast(d,bg,f,origin,ndraw=c['predictive_draws'],seed=rseed+900000+mi,out=out); forecasts.extend(fc.assign(scenario=s,unit=rep+1).to_dict('records'))
                        write_json(out/'completed.json',{'status':'complete','seed':rseed+1000*mi+int(origin*100)})
                    except Exception as e:
                        print('FAILED:',repr(e),flush=True); failures.append(dict(scenario=s,rep=rep+1,model=model,origin=origin,error=repr(e))); (out/'error.txt').write_text(traceback.format_exc())
                    # Save partial summaries after each fit, so a Colab interruption does not erase completed work.
                    pd.DataFrame(metrics).to_csv(root/'progress_metrics.csv',index=False); pd.DataFrame(recovery).to_csv(root/'progress_recovery.csv',index=False); pd.DataFrame(forecasts).to_csv(root/'progress_forecasts.csv',index=False); pd.DataFrame(failures).to_csv(root/'progress_failures.csv',index=False)
    if metrics: table(pd.DataFrame(metrics),root/'simulation_metrics')
    if recovery:
        r=pd.DataFrame(recovery); table(r,root/'all_parameter_recovery'); r=r[r.origin_day==min(c['origins'])].copy(); r['sq_error']=r.error**2; agg=r.groupby(['scenario','parameter']).agg(n=('error','size'),bias=('error','mean'),mse=('sq_error','mean'),coverage90=('covered90','mean')).reset_index(); agg['RMSE']=np.sqrt(agg.pop('mse')); agg['coverage_MCSE']=np.sqrt(agg.coverage90*(1-agg.coverage90)/agg.n); table(agg,root/'parameter_recovery_summary')
    summarize_scores(forecasts,root,'scenario'); table(pd.DataFrame(failures,columns=['scenario','rep','model','origin','error']),root/'failures'); return pd.DataFrame(metrics),pd.DataFrame(forecasts)

# -------------------- Real catalogue --------------------
CATALOGUE='https://cascadiaquakes.github.io/earthquake_catalog_repository/wech-jgr-2021/'
VIEWER='https://eqcat.cascadiaquakes.org/'
# Initial geometry chosen in advance as a working northern Cascadia window, not a verified fault boundary.
REAL_DEFAULT=dict(center_lat=48.5,center_lon=-123.5,strike_deg=330.,box=[[-150.,150.],[-40.,40.]],start='2017-01-01',end='2026-01-01',dt=.25,jitter_seconds=None,jitter_seed=20261125,drop_exact_duplicates=False,marked=False,source_url=VIEWER,coverage_confirmed=False)

def utc(x): return pd.Timestamp(x,tz='UTC') if pd.Timestamp(x).tzinfo is None else pd.Timestamp(x).tz_convert('UTC')

def read_catalogue(paths,root,options):
    root=Path(root); root.mkdir(parents=True,exist_ok=True); opt=REAL_DEFAULT|options; write_json(root/'catalogue_options.json',opt); dfs=[]; provenance=[]; audit=[]
    for path in map(Path,paths):
        if not path.exists(): raise FileNotFoundError(path)
        raw=pd.read_csv(path,low_memory=False); provenance.append(dict(file=path.name,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),rows=len(raw),columns=list(raw.columns),source=opt['source_url']))
        d=raw.copy(); d.columns=[str(k).strip().lower().replace(' ','_') for k in d.columns];
        def pick(*names): return next((n for n in names if n in d),None)
        lat=pick('latitude','lat','event_latitude'); lon=pick('longitude','lon','long','event_longitude'); ts=pick('origin_time','time','datetime','timestamp','date_time','event_time','utc_time','date'); mag=pick('magnitude','mag','mel','m_el','energy_magnitude')
        if lat is None or lon is None: raise ValueError(f'Missing latitude/longitude in {path.name}; columns: {list(d.columns)}')
        if all(k in d for k in ['year','month','day','hour','minute','second']):
            pieces=d[['year','month','day','hour','minute']].apply(pd.to_numeric,errors='coerce'); t=pd.to_datetime(pieces,errors='coerce',utc=True)+pd.to_timedelta(pd.to_numeric(d.second,errors='coerce'),unit='s')
        elif ts:
            if 'date' in d and 'time' in d and ts in ('date','time'): t=pd.to_datetime(d.date.astype(str)+' '+d.time.astype(str),utc=True,format='mixed',errors='coerce')
            else: t=pd.to_datetime(d[ts],utc=True,format='mixed',errors='coerce')
        else: raise ValueError(f'Missing timestamp or YEAR...SECOND in {path.name}; columns: {list(d.columns)}')
        # Preserve source records; identical full rows are dropped only with explicit opt-in.
        dup=raw.duplicated(keep='first'); audit.append({'stage':f'{path.name}: raw','rows':len(raw),'exact_duplicate_rows':int(dup.sum())}); keep=~dup if opt['drop_exact_duplicates'] else np.ones(len(raw),bool)
        q=pd.DataFrame(dict(timestamp=t,latitude=pd.to_numeric(d[lat],errors='coerce'),longitude=pd.to_numeric(d[lon],errors='coerce'),magnitude=pd.to_numeric(d[mag],errors='coerce') if mag else np.nan,source=path.name,row=np.arange(len(d))))
        dfs.append(q[keep]);
    write_json(root/'source_provenance.json',{'metadata':CATALOGUE,'viewer':VIEWER,'export_files':provenance,'user_confirmed_complete_export':opt['coverage_confirmed'],'time_assumption':'UTC (verify export metadata)'}); d=pd.concat(dfs,ignore_index=True); audit.append({'stage':'combined','rows':len(d),'missing_time':int(d.timestamp.isna().sum()),'missing_coordinates':int(d[['latitude','longitude']].isna().any(axis=1).sum()),'missing_magnitude':int(d.magnitude.isna().sum())});
    good=d.timestamp.notna()&d.latitude.between(-90,90)&d.longitude.between(-180,180); d=d[good].copy(); audit.append({'stage':'valid timestamps and coordinates','rows':len(d)})
    d=d[d.timestamp.between(utc(opt['start']),utc(opt['end']),inclusive='left')].copy(); audit.append({'stage':'calendar restriction','rows':len(d)}); 
    if d.empty: raise ValueError('No usable catalogue records within the requested period')
    center=CRS.from_proj4(f"+proj=aeqd +lat_0={opt['center_lat']} +lon_0={opt['center_lon']} +datum=WGS84 +units=m +no_defs"); trans=Transformer.from_crs('EPSG:4326',center,always_xy=True); xx,yy=trans.transform(d.longitude.to_numpy(),d.latitude.to_numpy()); theta=np.deg2rad(opt['strike_deg']); d['x_km']=(xx*np.sin(theta)+yy*np.cos(theta))/1000; d['y_km']=(xx*np.cos(theta)-yy*np.sin(theta))/1000
    box=np.asarray(opt['box'],float); selected=d.x_km.between(*box[0])&d.y_km.between(*box[1]); full=d.copy(); d=d[selected].copy(); audit.append({'stage':'fixed rotated domain','rows':len(d),'tied_timestamp_records':int(d.timestamp.duplicated(keep=False).sum()),'missing_magnitude':int(d.magnitude.isna().sum())}); table(pd.DataFrame(audit),root/'data_audit')
    if d.empty: raise ValueError('No detections in working rectangle; inspect region / projection')
    # Explicitly audit all year/month counts; zero is an exported count, not evidence of coverage.
    all_months=pd.date_range(utc(opt['start']),utc(opt['end']),freq='MS',inclusive='left'); monthly=d.set_index('timestamp').resample('MS').size().reindex(all_months,fill_value=0); yearly=d.assign(year=d.timestamp.dt.year).groupby('year').agg(events=('timestamp','size'),missing_magnitude=('magnitude',lambda x:x.isna().sum()),tied_records=('timestamp',lambda x:x.duplicated(keep=False).sum())).reindex(range(utc(opt['start']).year,utc(opt['end']).year),fill_value=0).reset_index(); table(yearly,root/'year_audit'); table(monthly.rename('events').rename_axis('month_utc').reset_index(),root/'monthly_audit')
    fig,ax=plt.subplots(1,2,figsize=(10,4.4),layout='constrained'); ax[0].scatter(full.longitude,full.latitude,s=1,alpha=.12,color='grey',rasterized=True); ax[0].scatter(d.longitude,d.latitude,s=1,alpha=.3,color='#246B8E',rasterized=True); ax[0].set(xlabel='Longitude (°)',ylabel='Latitude (°)',title='Catalogue and retained locations'); ax[0].set_aspect(1/np.cos(np.deg2rad(opt['center_lat']))); ax[1].scatter(d.x_km,d.y_km,s=1,alpha=.25,color='#246B8E',rasterized=True); ax[1].set(xlim=box[0],ylim=box[1],xlabel='Along-strike (km)',ylabel='Cross-strike (km)',title='Fixed local analysis rectangle'); ax[1].set_aspect('equal'); savefig(fig,root/'geographic_coverage')
    fig,ax=plt.subplots(figsize=(10,3.5),layout='constrained'); ax.plot(monthly.index,monthly.values,color='#246B8E',lw=1.1); ax.axvspan(utc('2023-01-01'),utc('2024-01-01'),color='#C28D38',alpha=.13,label='Validation year'); ax.axvspan(utc('2024-01-01'),utc('2026-01-01'),color='#186E66',alpha=.1,label='Test years'); ax.set(ylabel='Exported detections / month',title='Catalogue coverage and chronological periods'); ax.legend(); savefig(fig,root/'catalogue_monthly_counts')
    if not opt['coverage_confirmed']: raise ValueError('Inspect data audit and confirm the export is complete for the requested domain and period (set coverage_confirmed=True). Do not interpret export omissions as zero events.')
    if d.timestamp.duplicated().any() and not opt['jitter_seconds']: raise ValueError('Tied timestamps: verify reporting interval and set jitter_seconds explicitly for the working randomized-time sensitivity analysis; no ties are silently removed.')
    if opt['jitter_seconds']:
        rng=np.random.default_rng(opt['jitter_seed']); d['original_timestamp']=d.timestamp; d['timestamp']=d.timestamp+pd.to_timedelta(rng.uniform(0,opt['jitter_seconds'],len(d)),unit='s') # assumes floor-rounded interval, not exact interval-censored likelihood
        if d.timestamp.duplicated().any(): raise ValueError('Ties persist after jitter; inspect input resolution')
    d=d[d.timestamp.between(utc(opt['start']),utc(opt['end']),inclusive='left')].sort_values('timestamp').reset_index(drop=True); d.to_csv(root/'retained_catalogue.csv',index=False)
    train=d[d.timestamp<utc('2023-01-01')];
    if train.empty: raise ValueError('Need pre-2023 observations to construct training-only background; check data coverage.')
    # A deliberately modest fixed spatial basis, learned from pre-2023 data only.
    xy=train[['x_km','y_km']].to_numpy(); center_xy=np.median(xy,axis=0); widths=np.maximum(np.std(xy,axis=0),[5.,3.]); bg=Background(box,center_xy[None,:],widths[None,:],np.array([.2,.8])); threshold=None
    if opt['marked']:
        if d.magnitude.isna().any(): raise ValueError('Marked fit requires complete magnitudes in this implementation; use unmarked fit rather than remove events.')
        threshold=float(train.magnitude.median()); d['mark']=(d.magnitude>threshold).astype(int); K=2
    else: d['mark']=0; K=1
    write_json(root/'background_definition.json',{'box_km':box.tolist(),'centers_km':bg.centers.tolist(),'bandwidths_km':bg.widths.tolist(),'weights':bg.weights.tolist(),'training_end':'2023-01-01','marked':opt['marked'],'training_mark_threshold':threshold,'projection':center.to_string(),'strike_deg_clockwise_from_north':opt['strike_deg']}); return d,bg,K,opt

def real_windows(c,opt):
    # 30-day windows start every 30 days from Jan 1, within each year; no selection by event count.
    years=c.get('real_years',[2017,2018,2019,2020,2021,2022,2023,2024,2025]); n=c['real_windows_per_year']
    for y in years:
        for i in range(n):
            start=utc(f'{y}-01-01')+pd.Timedelta(days=30*i); end=start+pd.Timedelta(days=30)
            if start>=utc(opt['start']) and end<=utc(opt['end']): yield start,end,'training' if y<=2022 else 'validation' if y==2023 else 'test'

def run_real(paths,root,c,options=None,seed=84521):
    root=Path(root)/'real_data'; root.mkdir(parents=True,exist_ok=True); d,bg,K,opt=read_catalogue(paths,root,options or {}); rows=[]; failures=[]; windows=[]
    for i,(start,end,period) in enumerate(real_windows(c,opt)):
        unit=start.strftime('%Y%m%d'); w=root/period/unit; w.mkdir(parents=True,exist_ok=True); sub=d[d.timestamp.between(start,end,inclusive='left')].copy(); tt=(sub.timestamp-start).dt.total_seconds().to_numpy()/86400; # exactly boundary times require the documented jitter model
        if np.any(tt==0):
            # preserve the event with representable tiny positive offset; no event is discarded.
            tt=np.where(tt==0,1e-12,tt)
        data=Data(tt,sub[['x_km','y_km']].to_numpy(),sub.mark.to_numpy(),30.,opt['dt'],bg.box,K); save_data(data,bg,w); windows.append(dict(unit=unit,period=period,start_utc=str(start),end_utc=str(end),events=len(data.t))); print(f'\nReal {unit} | {period} | {len(data.t)} detections',flush=True)
        for origin in c['origins']:
            for mi,model in enumerate(c['models']):
                out=w/f'origin_{origin:g}'/model; out.mkdir(parents=True,exist_ok=True)
                try:
                    f,ss=fit_one(data.prefix(origin),bg,model,c,seed+10000*i+100*mi+int(origin),out); fc=forecast(data,bg,f,origin,ndraw=c['predictive_draws'],seed=seed+200000+100*i+mi,out=out); rows.extend(fc.assign(unit=unit,period=period,window_start=str(start),diagnostic_review=bool(((ss.r_hat>1.01)|(ss.ess_bulk<400)|~np.isfinite(ss.r_hat)).any())).to_dict('records')); write_json(out/'completed.json',{'status':'complete'})
                except Exception as e:
                    print('FAILED:',repr(e),flush=True); failures.append(dict(unit=unit,period=period,model=model,origin=origin,error=repr(e))); (out/'error.txt').write_text(traceback.format_exc())
                pd.DataFrame(rows).to_csv(root/'progress_forecasts.csv',index=False); pd.DataFrame(failures).to_csv(root/'progress_failures.csv',index=False)
    table(pd.DataFrame(windows),root/'windows'); table(pd.DataFrame(failures,columns=['unit','period','model','origin','error']),root/'failures'); summarize_scores(rows,root,'period'); return pd.DataFrame(rows)

def archive(root,download=False):
    root=Path(root); dst=Path(shutil.make_archive(str(root), 'zip',root_dir=root)); print(f'Results ZIP: {dst} ({dst.stat().st_size/1e6:.2f} MB)')
    if download:
        from google.colab import files
        files.download(str(dst))
    return dst


def simulation_calibration(root):
    """Create scenario-wise calibration tables and figures from completed M5 fits.
    Bins within a replicate are dependent; plotted points are descriptive.
    """
    root=Path(root)/'simulation'; rows=[]
    for p in root.glob('S*/rep_*/origin_25/M5/truth_state_comparison.csv'):
        d=pd.read_csv(p); d['scenario']=p.parts[-5]; d['unit']=p.parts[-4]; rows.append(d)
    if not rows: return
    df=pd.concat(rows,ignore_index=True); df['probability_bin']=pd.cut(df.p_pause,np.linspace(0,1,11),include_lowest=True); cal=df.groupby(['scenario','probability_bin'],observed=True).agg(n=('p_pause','size'),mean_predicted=('p_pause','mean'),observed_frequency=('true_pause','mean')).reset_index(); cal['probability_bin']=cal.probability_bin.astype(str); table(cal,root/'pause_calibration')
    fig,ax=plt.subplots(figsize=(6.5,5),layout='constrained'); ax.plot([0,1],[0,1],ls='--',color='gray',label='Ideal calibration');
    for s,g in cal.groupby('scenario'): ax.plot(g.mean_predicted,g.observed_frequency,marker='o',ms=4,label=s,alpha=.8)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean posterior probability',ylabel='True pause frequency',title='Pause-state calibration (descriptive)'); ax.legend(ncol=2); savefig(fig,root/'pause_calibration')
    p=root/'parameter_recovery_summary.csv'
    if p.exists():
        d=pd.read_csv(p); d=d[d.parameter.isin(['v','q1','q2','r1','r2','beta','alpha0','A_T','signed_km'])]; pv=d.pivot(index='parameter',columns='scenario',values='coverage90'); fig,ax=plt.subplots(figsize=(7,5),layout='constrained'); im=ax.imshow(pv.to_numpy(),aspect='auto',vmin=0,vmax=1,cmap='viridis'); ax.set(xticks=range(len(pv.columns)),xticklabels=pv.columns,yticks=range(len(pv.index)),yticklabels=pv.index,title='Empirical 90% interval coverage'); fig.colorbar(im,ax=ax,label='Coverage'); savefig(fig,root/'coverage_heatmap')


def run_sbc(root,c,nrep=20,seed=3381):
    """Optional simulation-based calibration under the *fitting* prior and same memory.
    Fixed background basis, one mark, 5 days, all other priors as fitted.
    Ranks with randomized ties; diagnostic only (autocorrelated draws affect SBC).
    """
    root=Path(root)/'sbc'; root.mkdir(parents=True,exist_ok=True); rng=np.random.default_rng(seed); par=Parameterization('M5',1,.25,c['learn_duration']); box=np.array([[-150.,150.],[-40.,40.]]); bg=Background(box,np.array([[0.,0.]]),np.array([[50.,12.]]),np.array([.2,.8])); ranks=[]; failures=[]
    for rep in range(nrep):
        try:
            x=[]
            for name,kind,a,b,lo,hi in par.spec:
                while True:
                    u=rng.normal(a,b) if kind=='log' else logit(rng.beta(a,b))
                    if lo<u<hi: x.append(u); break
            x=np.array(x); p=par.decode(x,np.ones(1)); z=path_sample(rng,20,p,'M5'); d=simulate(rng,5.,.25,box,bg,z,p,'M5',c['memory'],max_events=20000); out=root/f'rep_{rep+1:04d}'; save_data(d,bg,out); f,s=fit_one(d,bg,'M5',c,seed+rep+1000,out); draws=f['X'].reshape(-1,len(x)); ids=rng.choice(len(draws),size=min(len(draws),100),replace=False); draws=draws[ids]
            for j,name in enumerate(par.names):
                less=np.sum(draws[:,j]<x[j]); ties=np.sum(draws[:,j]==x[j]); ranks.append(dict(rep=rep+1,parameter=name,rank=int(less+rng.integers(ties+1)),draws=len(draws),scaled_rank=(less+.5)/(len(draws)+1)))
        except Exception as e: failures.append(dict(rep=rep+1,error=repr(e))); print('SBC failed:',e)
        pd.DataFrame(ranks).to_csv(root/'progress_ranks.csv',index=False)
    table(pd.DataFrame(failures,columns=['rep','error']),root/'failures')
    if ranks:
        r=pd.DataFrame(ranks); table(r,root/'ranks'); names=[k for k in ['v','sqrt_q1','beta','alpha0','b1','h0'] if k in r.parameter.unique()]; fig,axes=plt.subplots(len(names),1,figsize=(7,1.6*len(names)),layout='constrained',squeeze=False)
        for ax,k in zip(axes[:,0],names): ax.hist(r[r.parameter==k].scaled_rank,bins=np.linspace(0,1,11),color='#246B8E'); ax.set(ylabel=k,xlim=(0,1))
        axes[-1,0].set_xlabel('Scaled posterior rank'); fig.suptitle('Simulation-based calibration | exploratory'); savefig(fig,root/'rank_histograms')


## 3. Experiment controls
`SMOKE`: S2 and M1/M3/M5 only, tiny chains for verifying the workflow. `PILOT`: all eight scenarios and seven models, one replicate per scenario. `RESEARCH`: larger, still potentially insufficient chains, 20 replicates per scenario and two forecast origins. For the manuscript aim for 200 replicates per scenario in batches if feasible. Review convergence and increase chain lengths as necessary. Never use SMOKE or uncalibrated fits as publication results.

In [ ]:
from workflow import *
PROFILE = 'SMOKE'                 # 'PILOT' or 'RESEARCH' for substantive runs
RUN_NAME = 'dual_clock_smoke_001'  # Change name for every independent run/batch
OUTPUT_BASE = Path('/content') if Path('/content').exists() else Path.cwd()
RUN_SIMULATION = True
RUN_REAL = True                  # Set False to run simulation only
RUN_SBC = False                  # optional, expensive
RUN_SENSITIVITY = False          # optional, expensive
REP_START = 0                    # change when batching simulation replicates
SEED = 20261125
C = config(PROFILE)
# Examples of deliberate overrides:
# C.update(reps=5, steps=24000, burn=12000, thin=10, chains=4, predictive_draws=500)
# C.update(models=['M3','M5'], scenarios=['S1','S2','S3','S4'])
# C.update(memory=np.inf)        # exact full-memory fit, potentially expensive
# C.update(max_pairs=5_000_000)  # cost guard; never subsamples events
# Fixed real windows; smoke only exercises Jan 2024. Pilot/research use all years by default.
if PROFILE == 'SMOKE': C['real_years'] = [2024]
# For persistent storage in Colab: mount Google Drive and change OUTPUT_BASE before init_run.
ROOT = init_run(OUTPUT_BASE / RUN_NAME, C)
print(json.dumps(C,indent=2,default=str))


## 4. Simulation experiment
Exact conditional branching simulation generates S1–S8, including a no-pause case, intermittent transport, rate-only confounding, weak identification, detection loss, misspecified positive diffusion, and marked observations. Fits use the first 25 days (and 27 days in research mode). Forecasts use 6-, 24- and 72-hour horizons with no future observations in the fit. Numerical checks run at initialization. Diagnostic outputs do not prove adequate mixing.

In [ ]:
if RUN_SIMULATION:
    sim_metrics_df, sim_forecasts_df = run_simulation(ROOT, C, seed=SEED, rep_start=REP_START)
    simulation_calibration(ROOT)
    display(sim_metrics_df.head(12))
    display(sim_forecasts_df.head(12))
    archive(ROOT, download=False)  # intermediate checkpoint archive


## 5. Real catalogue: acquisition and audit
**Official source:** [CRESCENT Wech catalogue](https://cascadiaquakes.github.io/earthquake_catalog_repository/wech-jgr-2021/) / [viewer](https://eqcat.cascadiaquakes.org/) / [PNSN tremor](https://pnsn.org/tremor/). Select **Wech 2021 — Cascadia Tremor**, 2017–2025, and export CSV. Verify all records are exported (not sampled or capped); if necessary export non-overlapping time segments. Upload all CSVs or supply their paths. The loader accepts `origin_time, latitude, longitude, magnitude` or `YEAR, MONTH, DAY, HOUR, MINUTE, SECOND, LAT, LON, MAG` schemas. Original source checksums, audit counts, geometry and metadata are saved. Do not use a generic earthquake export. Source paper: [Wech (2021)](https://doi.org/10.1029/2021JB022523).

The initial geometry (48.5°N, −123.5°, 330° strike, ±150 × ±40 km rectangle) is provisional, not a verified geological boundary. Fix your domain using geological information/training observations, not held-out results. Verify completeness and UTC timestamps. First run with `coverage_confirmed=False` to obtain the audit. After verifying the data change to True and rerun the following real-data cell. If tied timestamps occur, verify the reporting interval and set `jitter_seconds` explicitly; this assumes floor rounding, a sensitivity approximation rather than exact interval-censored inference. Do not silently remove tied events. Keep the primary real analysis unmarked unless energy marks are complete and comparable.

In [ ]:
CATALOG_PATHS = []  # e.g. ['/content/wech_2017.csv', '/content/wech_2018.csv', ...]
DIRECT_CSV_URLS = [] # optional: only use a verified direct catalogue CSV URL, not the viewer URL
REAL_OPTIONS = dict(REAL_DEFAULT)
REAL_OPTIONS.update(
    coverage_confirmed=False,  # explicitly set True after confirming complete export and time standard
    jitter_seconds=None,      # e.g. 1.0 ONLY if source verifies 1-second floor rounding
    drop_exact_duplicates=False,
    marked=False,
    source_url=VIEWER,
)
if RUN_REAL and DIRECT_CSV_URLS:
    import requests
    for j,url in enumerate(DIRECT_CSV_URLS):
        dest=OUTPUT_BASE / f'catalogue_source_{j+1}.csv'
        with requests.get(url,stream=True,timeout=(15,120)) as resp:
            resp.raise_for_status()
            with dest.open('wb') as fp:
                for chunk in resp.iter_content(1024*1024): fp.write(chunk)
        CATALOG_PATHS.append(str(dest))
    REAL_OPTIONS['source_url']=DIRECT_CSV_URLS
if RUN_REAL and not CATALOG_PATHS:
    if importlib.util.find_spec('google.colab') is not None:
        from google.colab import files
        uploaded=files.upload()
        CATALOG_PATHS=[str(Path(name).resolve()) for name in uploaded if name.lower().endswith('.csv')]
    else:
        print('Set CATALOG_PATHS to the frozen CSV export(s). No real data substituted.')
print(CATALOG_PATHS)


## 6. Run the real-data analysis
This uses a within-window rolling forecast, not a shared cross-window posterior. In each fixed 30-day window the first 25/27 days inform the fit; the next 6/24/72 hours are evaluated. The fixed background basis and optional mark threshold are learned using pre-2023 data. Training (2017–22), validation (2023) and test (2024–25) outputs are labelled separately. Confirm outage-free windows or explicitly exclude unreliable windows before final inference. If the audit stops, change options in the next cell (or above), then rerun. The result archive will still contain the audit.

In [ ]:
# After verifying the data, uncomment and rerun this cell only:
# REAL_OPTIONS['coverage_confirmed'] = True
# REAL_OPTIONS['jitter_seconds'] = ... # verified reporting resolution, if needed
if RUN_REAL and CATALOG_PATHS:
    try:
        real_scores=run_real(CATALOG_PATHS, ROOT, C, REAL_OPTIONS, seed=SEED+123)
        display(real_scores.head(12))
    except Exception as exc:
        print('REAL ANALYSIS STOPPED:',exc)
        write_json(ROOT/'real_data'/'run_status.json', {'status':'stopped','error':repr(exc)})
    else:
        write_json(ROOT/'real_data'/'run_status.json', {'status':'runner_finished','note':'Inspect failures.csv and diagnostics before interpreting results.'})
    archive(ROOT, download=False)
elif RUN_REAL:
    print('No catalogue CSV selected; real analysis not executed.')
    write_json(ROOT/'real_data'/'run_status.json',{'status':'not_run','reason':'no CSV supplied'})


## 7. Optional robustness and calibration
Rerun the real data under alternative memory, grid resolution, rotation and jitter seeds, with fixed declared geometry/settings. Do not choose settings using test scores. Simulation-based calibration generates new data under the fitting prior, not the fixed simulation truth. These optional routines use substantial compute. Grid changes recalibrate mean duration priors on physical time; zero/slow transport are distinct.

In [ ]:
if RUN_SBC:
    run_sbc(ROOT, C, nrep=20, seed=SEED+8000)
if RUN_SENSITIVITY and RUN_REAL and CATALOG_PATHS and REAL_OPTIONS['coverage_confirmed']:
    variants=[('memory_short',{'memory':2.},{}),('memory_long',{'memory':6.},{}),('grid_3h',{}, {'dt':.125}),('grid_12h',{}, {'dt':.5}),('strike_320',{}, {'strike_deg':320.}),('strike_340',{}, {'strike_deg':340.})]
    if REAL_OPTIONS.get('jitter_seconds'): variants.append(('jitter_seed2',{}, {'jitter_seed':SEED+100}))
    for name,changes,opts in variants:
        cc=C|{'models':['M3','M5']}|changes; oo=REAL_OPTIONS|opts; rr=ROOT/'sensitivity'/name
        write_json(rr/'configuration.json',cc)
        try: run_real(CATALOG_PATHS,rr,cc,oo,seed=SEED+123)
        except Exception as exc: write_json(rr/'error.json',{'error':repr(exc)}); print(name,exc)


## 8. Export everything
The ZIP contains all generated CSV/LaTeX/HTML tables, PNG/PDF figures, posterior and predictive draws, source files, diagnostic/failure logs, input provenance and configuration. Save the archive before closing Colab. Send this ZIP back for manuscript-ready tables and interpretation. Do not share the raw full source export unless permitted; the ZIP includes retained catalogue data and model inputs.

In [ ]:
# Show a compact manifest and download. This cell can be run after an interrupted experiment.
from pathlib import Path
import shutil
print('Result files:',sum(1 for p in ROOT.rglob('*') if p.is_file()))
try:
    from google.colab import files
    ZIP = archive(ROOT, download=True)
except ImportError:
    ZIP = archive(ROOT, download=False)
print(ZIP)
